In [1]:
import pandas as pd

In [33]:
import pandas as pd

dsc_xl = pd.ExcelFile('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/DSC/2026-07-17/20260717-DSC-UAFW.xls')
dsc_og_df = pd.read_excel('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/DSC/2026-07-17/20260717-DSC-UAFW.xls', 
                          sheet_name="Ramp 10.00 °Cmin to 30.00 °C",
                          header =1
                          )
print(dsc_xl.sheet_names)
dsc_og_df
# dsc_og_df = dsc_og_df.rename(columns={'GUID': "col", "b2f4ff98-5721-4145-bc5f-4becc4a889ae": "values"})
# print(dsc_og_df.iloc[2005,1], dsc_og_df.iloc[2006,1])
# # print(len(dsc_og_df["col"]))
# print(dsc_og_df)

['Details', 'Ramp 10.00 °Cmin to 30.00 °C']


,Temperature,Heat Capacity (Normalized),Heat Flow (Normalized)
0,°C,J/(g.°C),W/g
1,-79.82,0,0.045216
2,-79.82,0,0.04524
3,-79.82,0,0.045175
4,-79.82,0,0.045119
...,...,...,...
6587,28.34,NaN,NaN
6588,28.35,NaN,NaN
6589,28.37,NaN,NaN
6590,28.39,NaN,NaN


# extract data fxn

In [ ]:

# if you want a df of the details, use:
# details_df = pd.read_excel("path_to_excel_file", sheet_name = "Details", header = 1)

def extract_data(path): 
    # Read in original file 
    xl = pd.ExcelFile(path) 
    all_sheets = xl.sheet_names 
    print(all_sheets)
    
    data_sheets = all_sheets[1:] 
    df_list = [] 
    
    for sheet in data_sheets: 
        sheet_df = pd.read_excel(path, sheet_name=sheet, header=1) 
        
        # extract the units from the first row of data (index 0)
        units = sheet_df.iloc[0].fillna('').astype(str).tolist()
        
        # combine old column names with the units
        new_columns = []
        for col, unit in zip(sheet_df.columns, units):
            clean_col = col.split('.')[0] if '.' in col else col
            if unit:
                new_columns.append(f"{clean_col} ({unit})")
            else:
                new_columns.append(clean_col)
                
        # assign the new combined names back to the dataframe columns
        sheet_df.columns = new_columns
        
        # drop units row from the data
        sheet_df = sheet_df.drop(index=0).reset_index(drop=True)
        
        # add source sheet column
        sheet_df['Source Sheet'] = sheet 
        df_list.append(sheet_df) 
        
    # Vertically stack all the sheets together into one final df 
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # convert to float
    # errors='coerce' turns any unconvertible text or bad data into NaN safely
    for col in combined_df.columns:
        if col != 'Source Sheet':
            combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
    
    # make sure source sheet is last column in final df
    cols = [col for col in combined_df.columns if col != 'Source Sheet'] + ['Source Sheet']
    combined_df = combined_df[cols]
    
    return combined_df